In [1]:
table_name = "dbo.sales"

StatementMeta(, e5932764-0097-466a-92ce-2c62e223c4ba, 3, Finished, Available, Finished, False)

In [2]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType,
    DoubleType
)

schema = StructType([
    StructField("SalesOrderNumber", StringType(), True),
    StructField("SalesOrderLineNumber", IntegerType(), True),
    StructField("OrderDate", DateType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("EmailAddress", StringType(), True),
    StructField("Item", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("TaxAmount", DoubleType(), True)
])


df = (
    spark.read
    .schema(schema)
    .option("header", "true")
    .option("dateFormat", "yyyy-MM-dd")
    .option("quote", '"')
    .option("escape", "\\")
    .csv("Files/new_data/*.csv")
)

df = (
    df
    .withColumn(
        "FirstName",
        F.split(F.col("CustomerName"), " ", 2).getItem(0)
    )
    .withColumn(
        "LastName",
        F.split(F.col("CustomerName"), " ", 2).getItem(1)
    )
)

df = (
    df
    .withColumn("Year", F.year("OrderDate").cast("int"))
    .withColumn("Month", F.month("OrderDate").cast("int"))
)

result_df = df.select(
    "SalesOrderNumber",
    "SalesOrderLineNumber",
    "OrderDate",
    "Year",
    "Month",
    "FirstName",
    "LastName",
    "EmailAddress",
    "Item",
    "Quantity",
    "UnitPrice",
    "TaxAmount"
)

(
    result_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

row_count = spark.table(table_name).count()

print(f"Table: {table_name}")
print(f"Total rows: {row_count}")

notebookutils.notebook.exit(str(row_count))


StatementMeta(, e5932764-0097-466a-92ce-2c62e223c4ba, 4, Finished, Available, Finished, False)

Table: dbo.sales
Total rows: 32718
ExitValue: 32718